# MotorAssistEnv — SFT + GRPO on Colab (T4, 2–3 hr budget)

**OpenEnv Hackathon Submission** · Closed-loop adaptive Deep Brain Stimulation (aDBS) agent for Parkinson's disease.

| | |
|---|---|
| Environment | HuggingFace Space — calibrated Fleming et al. (2023) biophysics, 3 tasks |
| Model | `unsloth/Qwen2.5-3B-Instruct` 4-bit + LoRA (r=16) |
| Recipe | **SFT warm-start (heuristic teacher) → GRPO** with TRL `GRPOTrainer` + vLLM colocate |
| Reward | grader score + dense per-step + JSON-format compliance |
| Curriculum | `easy` → `medium` → `hard` |
| Wallclock | ≈ 2.0–2.5 hr on Colab T4 |

All training logic — system prompt, rollout, reward, plotting, eval — is imported from **`parkinsons_Motor.train`**. The notebook is just glue: install → configure → smoke-test → SFT → sanity → GRPO → plots → push.

**Runtime** → Colab → Runtime → Change runtime type → **GPU (T4)**. Add `HF_TOKEN` to Colab Secrets (key icon).


## 1 · Install dependencies


In [39]:
%%capture
!pip install -q --upgrade uv

!pip install -qqq \
    "torch==2.8.0" "triton>=3.3.0" torchvision bitsandbytes "xformers==0.0.32.post2" \
    "unsloth_zoo[base]" \
    "unsloth[base]"

!pip install -qqq "openenv-core[core]>=0.2.0"
!pip install -qqq \
    "trl==0.29.0" \
    "vllm>=0.11.0" \
    "transformers>=4.57.1,<4.58" \
    "peft>=0.15,<1" \
    "accelerate>=1.13,<2" \
    "datasets" \
    "pydantic>=2,<3" \
    "scipy==1.13.1" \
    "matplotlib" "pandas" "huggingface_hub>=0.20.0"
print('Dependencies installed.')

## 2 · Hugging Face login


In [40]:
import os
from huggingface_hub import login, whoami

hf_token = None
for getter in (
    lambda: __import__('google.colab', fromlist=['userdata']).userdata.get('HF_TOKEN'),
    lambda: os.environ.get('HF_TOKEN'),
):
    try:
        hf_token = getter()
        if hf_token:
            break
    except Exception:
        pass
if not hf_token:
    import getpass
    hf_token = getpass.getpass('Enter your Hugging Face token (write scope, hidden): ').strip()
assert hf_token, 'HF_TOKEN required.'
login(token=hf_token, add_to_git_credential=True)
os.environ['HF_TOKEN'] = hf_token
print('Logged in as:', whoami()['name'])

Logged in as: virustechhacks


## 3 · Configuration

Every knob lives here. Defaults are tuned for **Colab T4 in ≤ 2.5 hr**: small SFT seed, modest GRPO budget, attention-only LoRA, 512-token completions (Qwen2.5 has no thinking-by-default so completions are short).


In [41]:
import pathlib

ON_COLAB = 'COLAB_GPU' in os.environ or os.path.isdir('/content')
BASE_DIR = pathlib.Path('/content') if ON_COLAB else pathlib.Path.cwd()
BASE_DIR.mkdir(parents=True, exist_ok=True)

# ── Environment (live HF Space) ────────────────────────────────────────────
ENV_URL     = 'https://virustechhacks-parkinsons-motor.hf.space'
ENV_REPO_ID = 'virustechhacks/parkinsons_Motor'

# ── Model + push target ────────────────────────────────────────────────────
MODEL_ID = 'unsloth/Qwen2.5-3B-Instruct'
HUB_REPO = 'your-name/dbs-grpo-qwen2.5-3b'   # change before push

# ── Curriculum ─────────────────────────────────────────────────────────────
# IMPORTANT: `make_rollout_func` returns ONE GRPO sample = ONE FULL EPISODE
# (concatenated prompt+completion tokens). The trainer then forwards that
# full episode tensor through the model for log-prob recompute.
# This requires `max_seq_length` to be large enough to fit the entire episode.
#
# To fit on a Colab T4 (16GB), we cap turns and `MAX_SEQ_LENGTH`:
#   1. Cap turns at 18 across the board (still emits `done=True` reliably
#      for `easy`/`hard`; `medium` gets a partial dense-reward signal which
#      is fine — GRPO learns equally well from per-step rewards mid-episode).
#   2. Set MAX_SEQ_LENGTH below to 16384 so 18 turns × ~850 tokens fits
#      comfortably inside Unsloth's pre-allocated buffers.
#
# This also cuts GRPO wallclock vs. longer episode lengths.
# We use SFT_MAX_TURNS_PER_TASK below (with the original episode lengths)
# to keep teacher rollouts long enough to actually emit `grader_score`.
TRAIN_TASKS = ['easy', 'medium', 'hard']
EVAL_TASKS  = ['easy', 'medium', 'hard']
MAX_TURNS_PER_TASK     = {'easy': 18, 'medium': 18, 'hard': 18}   # GRPO + eval
SFT_MAX_TURNS_PER_TASK = {'easy': 36, 'medium': 60, 'hard': 30}   # rejection sampling needs full episodes

# ── SFT (rejection-sampled heuristic teacher → warm-start) ─────────────────
SFT_TASKS             = ['easy', 'medium']
SFT_EPISODES_PER_TASK = 8        # 8 easy + 8 medium = 16 teacher rollouts
SFT_MIN_GRADER        = 0.50     # rejection threshold
SFT_EPOCHS            = 1
SFT_LR                = 2e-4
SFT_BATCH             = 2
SFT_GRAD_ACCUM        = 4
SFT_MAX_SEQ_LEN       = 1024

# ── GRPO ───────────────────────────────────────────────────────────────────
GRPO_EPISODES   = 24             # 8 easy + 10 medium + 6 hard
GRPO_GENERATIONS = 4              # group size — TRL requires ≥ 2; 4 is the sweet spot for T4
GRPO_LR         = 2e-6
GRPO_BETA       = 0.02
PER_DEVICE_BATCH = 1
GRAD_ACCUM       = 4
MAX_PROMPT_LENGTH      = 1024
# Qwen2.5-3B-Instruct has NO thinking-by-default, so the JSON action takes
# ~80-120 tokens. Capping at 256 (not 512) forces the model to COMMIT to
# the JSON instead of rambling. With a 512 cap, the model uses the budget
# to "explain its reasoning" first, runs out of tokens, never closes the
# JSON brace, parse_action() returns None, and every GRPO sample falls back
# to the same heuristic action -> reward_std=0 -> no advantage -> no learning
# (this is the GRPO collapse mode you'll see in the trainer table as
# clipped_ratio=1.0, reward=0, kl>0).
MAX_COMPLETION_LENGTH  = 256
# IMPORTANT: `make_rollout_func` returns prompt_ids/completion_ids that are the
# full episode CONCATENATED across turns (~18 × ~850 tokens ≈ 15.3k). The model
# then has to forward this whole tensor for GRPO's log-prob recompute, so
# `max_seq_length` MUST be sized to fit a full episode — NOT just one turn.
# Setting this too small triggers Unsloth's mask-slice path and crashes with
# "size of tensor a (43735) must match size of tensor b (1536)".
MAX_SEQ_LENGTH         = 16384   # = 16 × 1024; fits 18 turns × ~850 tokens with headroom

# ── Sampling ───────────────────────────────────────────────────────────────
# 0.7 keeps enough exploration for GRPO group diversity (group reward_std
# typically ~0.05-0.15 with this) while staying close to the SFT-trained
# distribution. Higher (0.9+) drifts the policy into garbage JSON; lower
# (0.5-) collapses group diversity to zero -> identical rewards -> no learning.
ROLLOUT_TEMPERATURE = 0.7
EVAL_TEMPERATURE    = 0.0
EVAL_SEEDS          = [101, 202, 303]

# ── LoRA ───────────────────────────────────────────────────────────────────
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.0
LORA_TARGETS = ['q_proj', 'k_proj', 'v_proj', 'o_proj']

SEED = 42
OUTPUT_DIR = BASE_DIR / 'artifacts' / 'motorassist'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import subprocess
try:
    print(subprocess.check_output(['nvidia-smi']).decode().split('\n')[0])
except Exception:
    print('nvidia-smi unavailable — enable a GPU runtime.')
print(f'Env:    {ENV_URL}')
print(f'Model:  {MODEL_ID}')
print(f'Output: {OUTPUT_DIR}')

Sun Apr 26 04:30:21 2026       
Env:    https://virustechhacks-parkinsons-motor.hf.space
Model:  unsloth/Qwen2.5-3B-Instruct
Output: /content/artifacts/motorassist


## 4 · Clone the env client + smoke-test the live Space

We clone the HF Space repo only for the typed dataclasses + the `parkinsons_Motor.train` helpers. The actual environment runs server-side.


In [42]:
import sys

ENV_CLONE_DIR = str(BASE_DIR / 'parkinsons_motor_space')
if not os.path.isdir(ENV_CLONE_DIR):
    print(f'Cloning {ENV_REPO_ID} ...')
    rc = subprocess.call(['git', 'clone', '--depth', '1',
                          f'https://huggingface.co/spaces/{ENV_REPO_ID}', ENV_CLONE_DIR])
    assert rc == 0, 'git clone failed (check ENV_REPO_ID / network).'
if ENV_CLONE_DIR not in sys.path:
    sys.path.insert(0, ENV_CLONE_DIR)
print('Env client cloned →', ENV_CLONE_DIR)

Env client cloned → /content/parkinsons_motor_space


In [43]:
import subprocess

client_file = f'{ENV_CLONE_DIR}/client.py'
print(f'Patching {client_file} to fix relative import...')

# Replace 'from .models' with 'from models' in client.py
# This addresses the 'ImportError: attempted relative import with no known parent package'
# because the cloned repo directory is added directly to sys.path, making its modules top-level.
subprocess.run(
    ['sed', '-i', 's/from .models import/from models import/', client_file],
    check=True
)
print('Patch applied successfully.')

Patching /content/parkinsons_motor_space/client.py to fix relative import...
Patch applied successfully.


In [44]:
import subprocess

train_file = f'{ENV_CLONE_DIR}/train.py'
print(f'Patching {train_file} to fix relative imports...')

# Replace 'from .client' with 'from client'
subprocess.run(
    ['sed', '-i', 's/from .client import/from client import/', train_file],
    check=True
)
# Replace 'from .core.models' with 'from models'
subprocess.run(
    ['sed', '-i', 's/from .core.models import/from models import/', train_file],
    check=True
)
print('Patch applied successfully to train.py.')

Patching /content/parkinsons_motor_space/train.py to fix relative imports...
Patch applied successfully to train.py.


In [45]:
import subprocess
import os

print(f'Patching all relative imports in {ENV_CLONE_DIR}...')

# Convert relative imports (dots) to absolute imports across all .py files
# Pattern: from ..[module] -> from [module]
subprocess.run(['find', ENV_CLONE_DIR, '-name', '*.py', '-exec', 'sed', '-i', 'r"s/from \.\.core/from core/g"', '{}', '+'], check=True)
subprocess.run(['find', ENV_CLONE_DIR, '-name', '*.py', '-exec', 'sed', '-i', 'r"s/from \.\.server/from server/g"', '{}', '+'], check=True)
subprocess.run(['find', ENV_CLONE_DIR, '-name', '*.py', '-exec', 'sed', '-i', 'r"s/from \.\.train/from train/g"', '{}', '+'], check=True)

# Pattern: from .[module] -> from [module]
subprocess.run(['find', ENV_CLONE_DIR, '-name', '*.py', '-exec', 'sed', '-i', 'r"s/from \.training/from training/g"', '{}', '+'], check=True)
subprocess.run(['find', ENV_CLONE_DIR, '-name', '*.py', '-exec', 'sed', '-i', 'r"s/from \.client/from client/g"', '{}', '+'], check=True)
subprocess.run(['find', ENV_CLONE_DIR, '-name', '*.py', '-exec', 'sed', '-i', 'r"s/from \.models/from models/g"', '{}', '+'], check=True)
subprocess.run(['find', ENV_CLONE_DIR, '-name', '*.py', '-exec', 'sed', '-i', 'r"s/from \.replay_grpo/from training.replay_grpo/g"', '{}', '+'], check=True)

print('Recursive patching complete. Retrying import...')

Patching all relative imports in /content/parkinsons_motor_space...
Recursive patching complete. Retrying import...


In [46]:
import asyncio
import nest_asyncio
nest_asyncio.apply()

from client import ParkinsonsMotorAction, ParkinsonsMotorEnv

async def _smoke():
    env = ParkinsonsMotorEnv(base_url=ENV_URL); await env.__aenter__()
    try:
        r = await env.reset(task_id='easy', seed=0); o = r.observation
        print(f'reset OK  beta={o.beta_arv:.3f}  tremor={o.tremor_arv:.3f}  force={o.force_preserved:.3f}')
        r = await env.step(ParkinsonsMotorAction(
            motor_command=o.target_output,
            dbs_amplitude=1.2, dbs_pulse_width=0.13, dbs_frequency=130.0,
        ))
        print(f'step OK   reward={r.reward:+.3f}  done={r.done}')
    finally:
        await env.__aexit__(None, None, None)

asyncio.run(_smoke())

reset OK  beta=0.854  tremor=0.076  force=0.654
step OK   reward=+0.666  done=False


## 5 · Import training utilities from `parkinsons_Motor`

All SFT/GRPO logic, reward functions, plot helpers and evaluation
utilities live in the `parkinsons_Motor.train` module so the notebook
stays focused on orchestration. The next cell does the actual import.


In [47]:
import sys
# Force reload if already attempted
if 'train' in sys.modules:
    import importlib
    importlib.reload(sys.modules['train'])

from train import (
    SYSTEM_PROMPT, TASK_CONTEXT, DEFAULT_REWARD_WEIGHTS,
    build_user_prompt, apply_chat_template,
    parse_action, make_action, heuristic_action,
    llm_generate, rollout_episode, rollout_episode_async, Trajectory,
    compute_reward, reward_total, reward_grader, reward_dense, reward_format,
    make_rollout_func, make_episode_logger,
    plot_training_dashboard, plot_training_loss,
    plot_baseline_vs_trained, compare_trajectories, save_training_plots,
    evaluate_model_on_task, evaluate_model_suite,
    eval_with_adapter_disabled, sanity_check_rollout,
)
print('System prompt (first 200 chars):')
print(SYSTEM_PROMPT[:200], '...')
print('Reward weights:', DEFAULT_REWARD_WEIGHTS)

System prompt (first 200 chars):
You are an expert closed-loop DBS controller managing Parkinsonian motor symptoms in real time.
Every step is a short clinical control decision: suppress pathological activity, preserve movement,
avoi ...
Reward weights: {'grader': 1.0, 'dense': 0.5, 'format': 0.2, 'invalid': 1.0}


## 6 · Load model + LoRA via Unsloth

Qwen2.5-3B-Instruct in 4-bit fits comfortably on a T4 (≈ 6 GB resident). LoRA on attention projections only — fastest possible adapter that still gives GRPO room to move.


In [48]:
import unsloth  # must come before transformers / trl
import torch, gc, glob
from unsloth import FastLanguageModel

# NOTE: We intentionally DO NOT call `PatchFastRL('GRPO', FastLanguageModel)`.
# That helper uses `inspect.getsource()` on TRL's GRPOTrainer, which fails on
# TRL >= 0.29 with "OSError: could not get source code" because the trainer
# class is dynamically built. Unsloth + TRL 0.29 work fine without it —
# `FastLanguageModel.get_peft_model` already gives GRPOTrainer everything it
# needs (LoRA adapter + gradient checkpointing + 4-bit base).

# ── Free any prior model from memory before reloading ──────────────────────
# Re-running this cell after `MAX_SEQ_LENGTH` was bumped requires a clean
# slate, otherwise Unsloth's cached attention buffers and any stale LoRA
# weights stay resident and corrupt the next forward pass with a
# device-side assert (the error is deferred — first cuda op after the
# bad kernel reports it, even if that op is innocuous like manual_seed).
for _name in ('trainer', 'sft_trainer', 'model'):
    if _name in globals():
        try:
            globals()[_name].cpu()
        except Exception:
            pass
        del globals()[_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

bf16  = bool(getattr(torch.cuda, 'is_bf16_supported', lambda: False)()) if torch.cuda.is_available() else False
dtype = torch.bfloat16 if bf16 else (torch.float16 if torch.cuda.is_available() else torch.float32)

# ── Load fresh base model ──────────────────────────────────────────────────
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_ID,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = dtype,
    load_in_4bit   = True,
)
if tokenizer.pad_token is None and tokenizer.eos_token is not None:
    tokenizer.pad_token = tokenizer.eos_token

# ── Decide cold vs warm path (do EXACTLY ONE) ──────────────────────────────
# Cold: no SFT adapter on disk -> apply a fresh LoRA via Unsloth.
# Warm: SFT adapter on disk -> load it via PeftModel.from_pretrained,
#       which builds the LoRA structure from the saved adapter_config.json
#       AND populates the weights in one go. This is the ONLY reload path
#       that doesn't try to do in-place adapter surgery on a quantized
#       base, which is what triggered the device-side assert before.
_sft_adapters = sorted(glob.glob(str(OUTPUT_DIR / 'run-*' / 'sft' / 'adapter')))
if _sft_adapters:
    from peft import PeftModel
    _latest = _sft_adapters[-1]
    model = PeftModel.from_pretrained(model, _latest, is_trainable=True)
    # Unsloth still needs to install its fast-RL hooks on the wrapped model.
    # `for_training` is a no-op on a fresh PEFT model but installs the
    # gradient-checkpointing-compat shim Unsloth requires.
    FastLanguageModel.for_training(model)
    print(f'Model loaded (WARM start from SFT adapter).  bf16={bf16}  dtype={dtype}')
    print(f'  ↳ adapter: {_latest}')
else:
    model = FastLanguageModel.get_peft_model(
        model,
        r              = LORA_R,
        target_modules = LORA_TARGETS,
        lora_alpha     = LORA_ALPHA,
        lora_dropout   = LORA_DROPOUT,
        bias           = 'none',
        use_gradient_checkpointing = True,
        random_state   = SEED,
    )
    print(f'Model loaded (COLD start, no SFT adapter on disk).  bf16={bf16}  dtype={dtype}')
    print('  ↳ section 7 will create the SFT adapter.')

==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 4.57.6. vLLM: 0.19.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Model loaded (WARM start from SFT adapter).  bf16=False  dtype=torch.float16
  ↳ adapter: /content/artifacts/motorassist/run-2026-04-26_03-34-27/sft/adapter


## 7 · SFT — rejection-sampled heuristic teacher → warm-start

Rolls `heuristic_action` (the clinical-priority baseline policy) against the live env on `easy` + `medium`, keeps only the steps from episodes whose **`grader_score ≥ SFT_MIN_GRADER`**, and writes them to `sft_data.jsonl` as `{system, user, assistant}` triples. SFT then pre-conditions the LLM on the JSON action format + clinical priors so GRPO spends its gradient budget on *optimization* not format learning.


### 7a · Generate SFT data via heuristic rejection sampling


In [49]:
import json, asyncio
from datetime import datetime

run_dir = OUTPUT_DIR / f'run-{datetime.now():%Y-%m-%d_%H-%M-%S}'
run_dir.mkdir(parents=True, exist_ok=True)
sft_jsonl = run_dir / 'sft_data.jsonl'

async def _collect_sft():
    env = ParkinsonsMotorEnv(base_url=ENV_URL); await env.__aenter__()
    examples, kept_eps, total_eps = [], 0, 0
    try:
        for task_id in SFT_TASKS:
            for ep in range(SFT_EPISODES_PER_TASK):
                seed = ep * 7 + 1
                r = await env.reset(task_id=task_id, seed=seed)
                obs = r.observation
                ep_examples, history, last_amp = [], [], None
                # SFT runs the FULL canonical episode so terminal `grader_score`
                # actually fires — GRPO uses the shorter MAX_TURNS_PER_TASK cap.
                max_turns = SFT_MAX_TURNS_PER_TASK.get(task_id, 36)
                grader = 0.0
                for step_idx in range(max_turns):
                    # Package signature: build_user_prompt(step, obs, task_id, history)
                    user_prompt = build_user_prompt(step_idx, obs, task_id, history)
                    action = heuristic_action(obs, task_id, last_amp=last_amp)
                    last_amp = action.dbs_amplitude
                    action_dict = {
                        'motor_command':   action.motor_command,
                        'dbs_amplitude':   action.dbs_amplitude,
                        'dbs_pulse_width': action.dbs_pulse_width,
                        'dbs_frequency':   action.dbs_frequency,
                    }
                    ep_examples.append({
                        'system':    SYSTEM_PROMPT,
                        'user':      user_prompt,
                        'assistant': json.dumps(action_dict),
                    })
                    step = await env.step(action)
                    # Update history with a one-line summary so the next prompt
                    # gets the same RECENT block the GRPO rollout will see at
                    # inference time (keeps SFT and GRPO prompt distributions aligned).
                    history.append(
                        f's{step_idx} amp={action.dbs_amplitude:.2f}mA '
                        f'pw={action.dbs_pulse_width:.2f} fq={action.dbs_frequency:.0f}Hz '
                        f'r={step.reward:+.2f}'
                    )
                    obs = step.observation
                    # The server emits `grader_score` as an attribute on the
                    # observation (-1.0 sentinel during the episode, real
                    # [0,1] score on the terminal step when scoring runs).
                    g = getattr(obs, 'grader_score', None)
                    if g is not None and float(g) >= 0.0:
                        grader = float(g)
                    if step.done:
                        break
                total_eps += 1
                if grader >= SFT_MIN_GRADER:
                    examples.extend(ep_examples)
                    kept_eps += 1
                    print(f'  [keep] {task_id} seed={seed} grader={grader:.3f}  steps={len(ep_examples)}')
                else:
                    print(f'  [drop] {task_id} seed={seed} grader={grader:.3f}')
    finally:
        await env.__aexit__(None, None, None)
    return examples, kept_eps, total_eps

examples, kept_eps, total_eps = asyncio.run(_collect_sft())
with open(sft_jsonl, 'w') as f:
    for ex in examples:
        f.write(json.dumps(ex) + '\n')
print(f'\nSFT rejection sampling: kept {kept_eps}/{total_eps} episodes, {len(examples)} steps')
print(f'Saved → {sft_jsonl}')

  [keep] easy seed=1 grader=0.776  steps=36
  [keep] easy seed=8 grader=0.771  steps=36
  [keep] easy seed=15 grader=0.709  steps=36
  [keep] easy seed=22 grader=0.723  steps=36
  [keep] easy seed=29 grader=0.713  steps=36
  [keep] easy seed=36 grader=0.632  steps=36
  [keep] easy seed=43 grader=0.646  steps=36
  [keep] easy seed=50 grader=0.720  steps=36
  [keep] medium seed=1 grader=0.514  steps=60
  [drop] medium seed=8 grader=0.492
  [drop] medium seed=15 grader=0.473
  [keep] medium seed=22 grader=0.500  steps=60
  [drop] medium seed=29 grader=0.498
  [drop] medium seed=36 grader=0.458
  [keep] medium seed=43 grader=0.522  steps=60
  [drop] medium seed=50 grader=0.473

SFT rejection sampling: kept 11/16 episodes, 468 steps
Saved → /content/artifacts/motorassist/run-2026-04-26_04-31-09/sft_data.jsonl


### 7b · SFT training (TRL `SFTTrainer`)


In [50]:
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

def _to_chat(ex):
    text = tokenizer.apply_chat_template(
        [
            {'role': 'system',    'content': ex['system']},
            {'role': 'user',      'content': ex['user']},
            {'role': 'assistant', 'content': ex['assistant']},
        ],
        tokenize=False, add_generation_prompt=False,
    )
    return {'text': text}

raw_ds = Dataset.from_list(examples)
sft_ds = raw_ds.map(_to_chat, remove_columns=raw_ds.column_names)
print(f'SFT dataset: {len(sft_ds)} examples')

sft_dir = run_dir / 'sft'
sft_dir.mkdir(parents=True, exist_ok=True)

sft_cfg = SFTConfig(
    output_dir                  = str(sft_dir),
    num_train_epochs            = SFT_EPOCHS,
    learning_rate               = SFT_LR,
    per_device_train_batch_size = SFT_BATCH,
    gradient_accumulation_steps = SFT_GRAD_ACCUM,
    max_seq_length              = SFT_MAX_SEQ_LEN,
    logging_steps               = 5,
    save_strategy               = 'no',
    bf16                        = bf16,
    fp16                        = torch.cuda.is_available() and not bf16,
    report_to                   = 'none',
    dataset_text_field          = 'text',
    packing                     = False,
)
sft_trainer = SFTTrainer(
    model           = model,
    processing_class= tokenizer,
    args            = sft_cfg,
    train_dataset   = sft_ds,
)
sft_trainer.train()
model.save_pretrained(str(sft_dir / 'adapter'))
tokenizer.save_pretrained(str(sft_dir / 'adapter'))
print(f'SFT done. Adapter → {sft_dir / "adapter"}')
if sft_trainer.state.log_history:
    losses = [x['loss'] for x in sft_trainer.state.log_history if 'loss' in x]
    if losses:
        print(f'final SFT loss = {losses[-1]:.4f}')

Map:   0%|          | 0/468 [00:00<?, ? examples/s]

SFT dataset: 468 examples


Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/468 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 468 | Num Epochs = 1 | Total steps = 59
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 7,372,800 of 3,093,311,488 (0.24% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
5,0.161300
10,0.148900
15,0.138000
20,0.131900
25,0.126100
30,0.122700
35,0.120400
40,0.121100
45,0.111800
50,0.114600


SFT done. Adapter → /content/artifacts/motorassist/run-2026-04-26_04-31-09/sft/adapter
final SFT loss = 0.1141


## 8 · Sanity check (post-SFT) — fail FAST not silently

One short rollout. Verifies that the **post-SFT** model still produces parseable JSON, that the env returns rewards, and that the WebSocket transport survives a long generation. Raises on failure so you don't burn 90 min of GPU on a broken loop.


In [51]:
_ = sanity_check_rollout(
    model, tokenizer, ENV_URL,
    task_id            = 'easy',
    seed               = 0,
    max_turns          = 4,
    temperature        = ROLLOUT_TEMPERATURE,
    max_new_tokens     = MAX_COMPLETION_LENGTH,
    max_prompt_length  = MAX_PROMPT_LENGTH,
    warm_up            = True,
    retry_on_env_error = True,
    raise_on_failure   = True,
)

[warm-up] compiling CUDA kernels with one throwaway generation ...

=== sanity_check_rollout  task=easy  seed=0  turns=4 ===

[trace] generating one completion offline (no env call) to inspect raw text ...
  completion tokens       : 45 / 256  (ok)
  has <think> ... </think>: open=False  close=False
  has any JSON-like {...}: True
  parse_action result     : {'motor_command': 0.0, 'dbs_amplitude': 1.2, 'dbs_pulse_width': 0.13, 'dbs_frequency': 130.0}
  --- raw completion (preview) ---
  | {"motor_command": 0.0, "dbs_amplitude": 1.2, "dbs_pulse_width": 0.13, "dbs_frequency": 130.0}
  --- end preview ---
  steps run               : 4 / 4
  parseable JSON          : 4 / 4  (100%)
  invalid_count           : 0
  steps with non-0 reward : 4 / 4
  dense reward (mean)     : +0.7522
  grader_score            : 0.0000
  episode_success         : False
  env_error               : None
  first step trace        : step=1 amp=1.20 => beta=0.532 tremor=0.061 se=0.074 r=+0.67
  last  step trace      

## 9 · GRPO setup — `make_rollout_func` + `make_episode_logger`

Schedule: **8 easy + 10 medium + 6 hard = 24 episodes**, each replicated `GRPO_GENERATIONS=4` times so a GRPO group sees identical env conditions with different policy samples. That's the kube-sre-gym trick that keeps group-relative advantages clean.


In [52]:
import random, logging
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

rng = random.Random(SEED)
schedule = (
    [('easy',   rng.randint(0, 99)) for _ in range(8)]
    + [('medium', rng.randint(0, 99)) for _ in range(10)]
    + [('hard',   rng.randint(0, 99)) for _ in range(6)]
)
rng.shuffle(schedule)
assert len(schedule) == GRPO_EPISODES, f'{len(schedule)} != {GRPO_EPISODES}'

train_dataset = Dataset.from_list([
    {'prompt': f"Manage DBS for a Parkinson's patient on task `{tid}` (seed={s}).",
     'task_id': tid, 'seed': s}
    for tid, s in schedule
])

reward_csv  = run_dir / 'reward_log.csv'
log_episode = make_episode_logger(reward_csv)
rollout_func = make_rollout_func(
    env_url            = ENV_URL,
    episodes           = schedule,
    max_turns_per_task = MAX_TURNS_PER_TASK,
    num_generations    = GRPO_GENERATIONS,
    log_episode        = log_episode,
    temperature        = ROLLOUT_TEMPERATURE,
    max_new_tokens     = MAX_COMPLETION_LENGTH,
    max_prompt_length  = MAX_PROMPT_LENGTH,
)

grpo_dir = run_dir / 'grpo'
grpo_dir.mkdir(parents=True, exist_ok=True)

grpo_config = GRPOConfig(
    output_dir                  = str(grpo_dir),
    num_train_epochs            = 1,
    learning_rate               = GRPO_LR,
    per_device_train_batch_size = PER_DEVICE_BATCH,
    gradient_accumulation_steps = GRAD_ACCUM,
    num_generations             = GRPO_GENERATIONS,
    generation_batch_size       = GRPO_GENERATIONS,
    # NOTE: `max_prompt_length` is omitted on purpose — Unsloth's
    # UnslothGRPOTrainer shim wraps an older GRPOConfig that doesn't accept
    # it as a kwarg (TypeError: unexpected keyword 'max_prompt_length').
    # The prompt cap is enforced by `make_rollout_func(... max_prompt_length=...)`
    # above, so dropping it here changes nothing functionally.
    max_completion_length       = MAX_COMPLETION_LENGTH,
    temperature                 = ROLLOUT_TEMPERATURE,
    # top_p MUST stay at 0.95 (nucleus sampling). Setting top_p=1.0 lets
    # the sampler pick rare tokens that derail JSON generation - the model
    # never closes the brace, parse_action returns None, every group sample
    # falls back to the same heuristic action -> reward_std=0 -> NO LEARNING.
    top_p                       = 0.95,
    # repetition_penalty via generation_kwargs (forwarded to GenerationConfig).
    # 1.05 discourages the "explain forever, never close JSON" pathology
    # without distorting the action distribution. Using the dict path instead
    # of a top-level kwarg insulates us from Unsloth shim version drift.
    generation_kwargs           = {'repetition_penalty': 1.05},
    beta                        = GRPO_BETA,
    scale_rewards               = True,
    # NOTE: vLLM is OFF on Colab T4 because vLLM 0.11's 4-bit bitsandbytes
    # path crashes under torch.compile with:
    #   "TypeError: getitem expected 2 arguments, got 3" inside
    #   bitsandbytes.py::_apply_4bit_weight on Qwen2 attention.
    # Falling back to HF generation costs ~1.3x throughput but keeps the
    # whole pipeline working on T4. (Re-enable on L4/A100 with bf16 base.)
    use_vllm                    = False,
    logging_steps               = 1,
    save_strategy               = 'steps',
    save_steps                  = 10,
    bf16                        = bf16,
    fp16                        = torch.cuda.is_available() and not bf16,
    gradient_checkpointing      = True,
    gradient_checkpointing_kwargs = {'use_reentrant': False},
    report_to                   = 'none',
    remove_unused_columns       = False,
    save_total_limit            = 1,
)

trainer = GRPOTrainer(
    model            = model,
    processing_class = tokenizer,
    reward_funcs     = [reward_total, reward_grader, reward_dense, reward_format],
    args             = grpo_config,
    train_dataset    = train_dataset,
    rollout_func     = rollout_func,
)
for attr in ('image_token_id', 'vision_start_token_id', 'vision_end_token_id'):
    if not hasattr(trainer, attr):
        setattr(trainer, attr, None)
print(f'GRPOTrainer ready.  CSV log → {reward_csv}')
print(f'Schedule (first 8): {schedule[:8]}')

GRPOTrainer ready.  CSV log → /content/artifacts/motorassist/run-2026-04-26_04-31-09/reward_log.csv
Schedule (first 8): [('medium', 13), ('easy', 94), ('medium', 86), ('medium', 4), ('medium', 54), ('hard', 11), ('easy', 14), ('hard', 27)]


/content/unsloth_compiled_cache/UnslothGRPOTrainer.py:4877: UserWarning: You are using 'rollout_func', which is an experimental feature. This API may change or be removed at any time without prior notice. Silence this warning by setting environment variable TRL_EXPERIMENTAL_SILENCE=1.
  super().__init__(


## 9b · Critical patch — make `rollout_func` actually fire under `use_vllm=False`

TRL v0.27 + Unsloth silently ignore `rollout_func` in the regular HF
generation path. We monkey-patch `trainer._generate_single_turn` to
honour our env-driven rollout, then smoke-test the patch with a stub
rollout to confirm it fires before `trainer.train()` is called.

Tracking issue: [`unslothai/unsloth#3573`](https://github.com/unslothai/unsloth/issues/3573).


In [ ]:
# ── Force `rollout_func` to actually run during GRPO (use_vllm=False) ─────
# WHY THIS CELL EXISTS:
#   TRL v0.27 / Unsloth's compiled GRPOTrainer SILENTLY IGNORES `rollout_func`
#   when use_vllm=False (see unslothai/unsloth#3573 — open as of Apr 2026).
#   In the regular HF-generation path, the trainer feeds raw dataset prompts
#   ("Manage DBS for a Parkinson's patient on task `easy`...") directly into
#   `model.generate()` WITHOUT applying our SYSTEM_PROMPT or running any
#   multi-turn env interaction. JSON parsing fails on every sample, every
#   reward function returns 0, the group reward_std is 0, there is no
#   advantage to learn from. That is exactly the "all zeros" GRPO collapse
#   you've been seeing in the trainer table:
#       reward=0  reward_std=0  clipped_ratio=1.0  mean_terminated_length=0
#
# WHAT THIS DOES:
#   Override `trainer._generate_single_turn` at the instance level so that
#   when our `rollout_func` is set, it gets called as the source of truth
#   for prompt_ids/completion_ids/logprobs (and forwards the per-episode
#   reward_total / reward_grader / reward_dense / reward_format / etc. as
#   `extra_fields`, exactly as TRL's vLLM path already does).
#
# WHEN TO REMOVE THIS CELL:
#   Once unslothai/unsloth#3573 is closed AND you've upgraded both `unsloth`
#   and `trl` to a version that honors `rollout_func` in the non-vLLM path.
import types

assert hasattr(trainer, 'rollout_func') and trainer.rollout_func is not None, (
    'trainer.rollout_func is unset — re-run the previous cell. Without it '
    'this patch has nothing to call.'
)
assert hasattr(trainer, '_generate_single_turn'), (
    "trainer has no `_generate_single_turn` method (TRL too old?). This patch "
    "was written against TRL >= 0.27 / Unsloth's compiled GRPOTrainer."
)

_orig_generate_single_turn = trainer.__class__._generate_single_turn

def _patched_generate_single_turn(self, prompts):
    # Only redirect when rollout_func is set AND we're on the regular HF
    # generation path. vLLM/paged paths in TRL v0.27 already consult
    # rollout_func directly, so leave them alone.
    if (self.rollout_func is not None
            and not getattr(self, 'use_vllm', False)
            and not getattr(self, 'use_transformers_paged', False)):
        output = self.rollout_func(list(prompts), self)
        required = {'prompt_ids', 'completion_ids', 'logprobs'}
        extra_fields = {k: v for k, v in output.items() if k not in required}
        return (
            output['prompt_ids'],
            output['completion_ids'],
            output['logprobs'],
            extra_fields,
        )
    return _orig_generate_single_turn(self, prompts)

trainer._generate_single_turn = types.MethodType(_patched_generate_single_turn, trainer)
print('[patch] _generate_single_turn override installed.')

# ── Smoke test the patch with a STUB rollout_func ─────────────────────────
# We swap in a dummy rollout_func that returns trivial fake ids, call the
# patched method once, and confirm the stub was invoked. This proves the
# override is wired correctly WITHOUT consuming a real schedule slot from
# the real `rollout_func` (which would skew the very first GRPO step).
_real_rollout = trainer.rollout_func
_stub_was_called = {'fired': False}

def _stub_rollout_func(prompts, _trainer):
    _stub_was_called['fired'] = True
    fake_ids = tokenizer('hello', return_tensors='pt').input_ids[0].tolist()
    n = len(prompts)
    return {
        'prompt_ids':      [fake_ids               for _ in range(n)],
        'completion_ids':  [fake_ids               for _ in range(n)],
        'logprobs':        [[0.0] * len(fake_ids)  for _ in range(n)],
        'reward_total':    [0.0] * n,
        'reward_grader':   [0.0] * n,
        'reward_dense':    [0.0] * n,
        'reward_format':   [0.0] * n,
        'grader_score':    [0.0] * n,
        'episode_success': [0]   * n,
        'n_steps':         [0]   * n,
    }

trainer.rollout_func = _stub_rollout_func
try:
    _pid, _cid, _lp, _ef = trainer._generate_single_turn([train_dataset[0]['prompt']])
    assert _stub_was_called['fired'], 'patch did NOT call rollout_func — patch is broken'
    assert isinstance(_pid, list) and len(_pid) == 1, f'unexpected prompt_ids shape: type={type(_pid).__name__} len={len(_pid)}'
    assert isinstance(_cid, list) and len(_cid) == 1, f'unexpected completion_ids shape: type={type(_cid).__name__} len={len(_cid)}'
    assert len(_cid[0]) > 0, 'stub returned empty completion_ids — patch broken'
    assert 'reward_total' in _ef, f'extra_fields missing reward_total; got keys={list(_ef.keys())}'
    print(f'[patch] smoke-test OK · prompt_tokens={len(_pid[0])} completion_tokens={len(_cid[0])} '
          f'extra_field_keys={sorted(_ef.keys())}')
finally:
    trainer.rollout_func = _real_rollout

print('[patch] ready — `trainer.train()` will now drive multi-turn env episodes.')


## 10 · Train


In [54]:
print('Starting GRPO training ...')
print(f'  tasks       : {TRAIN_TASKS}')
print(f'  episodes    : {GRPO_EPISODES}')
print(f'  generations : {GRPO_GENERATIONS}  (group size)')
print(f'  env URL     : {ENV_URL}')
print()
trainer.train()
trainer.save_model(str(grpo_dir / 'adapter'))
tokenizer.save_pretrained(str(grpo_dir / 'adapter'))
print(f'\nSaved final adapter → {grpo_dir / "adapter"}')

Starting GRPO training ...
  tasks       : ['easy', 'medium', 'hard']
  episodes    : 24
  generations : 4  (group size)
  env URL     : https://virustechhacks-parkinsons-motor.hf.space



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 24 | Num Epochs = 1 | Total steps = 24
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 7,372,800 of 3,093,311,488 (0.24% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / reward_total / mean,rewards / reward_total / std,rewards / reward_grader / mean,rewards / reward_grader / std,rewards / reward_dense / mean,rewards / reward_dense / std,rewards / reward_format / mean,rewards / reward_format / std
1,0.017200,0.000000,0.000000,256.000000,256.000000,256.000000,1.000000,0.000000,0.000000,0.000000,0.859394,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.009200,0.000000,0.000000,256.000000,256.000000,256.000000,1.000000,0.000000,0.000000,0.000000,0.460602,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


KeyboardInterrupt: 

## 11 · Reward & loss curves

`save_training_plots` writes the two PNGs the judges look for:
- **`training_dashboard.png`** — total / grader / per-task reward curves + reward decomposition
- **`training_loss.png`** — policy loss / mean reward / KL / grad-norm from `trainer.state.log_history`


In [ ]:
from IPython.display import Image, display

plot_paths = save_training_plots(
    reward_csv, run_dir,
    train_tasks = TRAIN_TASKS,
    log_history = trainer.state.log_history,
)
for name, path in plot_paths.items():
    print(f'{name}: {path}')
    display(Image(filename=str(path)))

## 12 · Evaluation — base vs trained on `easy / medium / hard`

Three seeds per task at `temperature=0` (deterministic). Same model, same seeds, only LoRA adapter toggled. Outputs `eval_baseline.json`, `eval_trained.json`, `eval_comparison.png`.


In [ ]:
import json as _json

eval_kwargs = dict(
    tasks              = EVAL_TASKS,
    seeds              = EVAL_SEEDS,
    max_turns_per_task = MAX_TURNS_PER_TASK,
    temperature        = EVAL_TEMPERATURE,
    max_new_tokens     = MAX_COMPLETION_LENGTH,
    max_prompt_length  = MAX_PROMPT_LENGTH,
)

print('Evaluating BASE model (LoRA disabled) ...')
baseline_results = eval_with_adapter_disabled(model, tokenizer, ENV_URL, **eval_kwargs)
print('\nEvaluating TRAINED model (LoRA active) ...')
trained_results  = evaluate_model_suite(model, tokenizer, ENV_URL, **eval_kwargs)

def _summary(label, results):
    print(f'\n--- {label} ---')
    for r in results:
        print(f"  {r['task_id']:6s}  mean={r['mean_score']:.3f} ± {r['std_score']:.3f}  "
              f"pass={r['pass_rate']*100:3.0f}%  amp={r['mean_amp_ma']:.2f} mA")
_summary('BASE', baseline_results); _summary('TRAINED', trained_results)

print('\n--- DELTA (trained − base) ---')
base_by = {r['task_id']: r for r in baseline_results}
for r in trained_results:
    b = base_by.get(r['task_id'], {})
    d_score = r['mean_score']     - float(b.get('mean_score', 0.0))
    d_pass  = r['pass_rate']*100  - float(b.get('pass_rate', 0.0))*100
    arrow   = '↑' if d_score > 0 else ('↓' if d_score < 0 else '·')
    print(f"  {r['task_id']:6s}  Δscore={d_score:+.3f} {arrow}   Δpass={d_pass:+5.0f} pts")

def _strip(results):
    return [{k: v for k, v in r.items() if k != 'rollouts'} for r in results]

(run_dir / 'eval_baseline.json').write_text(_json.dumps(_strip(baseline_results), indent=2))
(run_dir / 'eval_trained.json' ).write_text(_json.dumps(_strip(trained_results),  indent=2))
print(f'\nSaved {run_dir / "eval_baseline.json"}')
print(f'Saved {run_dir / "eval_trained.json"}')

### 12a · Comparison plot — base vs trained on the same axes


In [ ]:
comparison_png = plot_baseline_vs_trained(
    baseline_results, trained_results,
    run_dir / 'eval_comparison.png',
)
print(f'Saved {comparison_png}')
display(Image(filename=str(comparison_png)))

## 13 · Sample trajectory — before vs after on the same seed

Quantitative scores tell *that* the agent improved; this plot shows *how*. One fixed `(task, seed)`, rolled out twice (LoRA off, LoRA on), overlaying DBS amplitude / β-band / tremor / side-effect load.


In [ ]:
DEMO_TASK  = EVAL_TASKS[0] if EVAL_TASKS else 'easy'
DEMO_SEED  = int(EVAL_SEEDS[0]) if EVAL_SEEDS else 0
DEMO_TURNS = MAX_TURNS_PER_TASK.get(DEMO_TASK, 20)

print(f'Rolling task=`{DEMO_TASK}` seed={DEMO_SEED} for {DEMO_TURNS} turns × 2 (base, trained) ...')

with model.disable_adapter():
    base_traj = rollout_episode(
        model, tokenizer, ENV_URL,
        task_id=DEMO_TASK, seed=DEMO_SEED, max_turns=DEMO_TURNS,
        temperature=EVAL_TEMPERATURE,
        max_new_tokens=MAX_COMPLETION_LENGTH,
        max_prompt_length=MAX_PROMPT_LENGTH,
    )
trained_traj = rollout_episode(
    model, tokenizer, ENV_URL,
    task_id=DEMO_TASK, seed=DEMO_SEED, max_turns=DEMO_TURNS,
    temperature=EVAL_TEMPERATURE,
    max_new_tokens=MAX_COMPLETION_LENGTH,
    max_prompt_length=MAX_PROMPT_LENGTH,
)
print(f'  base    -> grader={base_traj.grader_score:.3f}  success={base_traj.episode_success}')
print(f'  trained -> grader={trained_traj.grader_score:.3f}  success={trained_traj.episode_success}')

trajectory_png = compare_trajectories(
    base_traj, trained_traj,
    run_dir / f'trajectory_compare_{DEMO_TASK}_seed{DEMO_SEED}.png',
)
print(f'Saved {trajectory_png}')
display(Image(filename=str(trajectory_png)))

## 14 · Push to Hugging Face Hub (optional)

Uncomment after editing `HUB_REPO` in cell 3.


In [ ]:
# trainer.push_to_hub(repo_id=HUB_REPO, commit_message='SFT + GRPO adapter for MotorAssistEnv (T4 budget run)')
# print(f'Pushed → https://huggingface.co/{HUB_REPO}')
print('Push step is commented out by default — uncomment when HUB_REPO is set.')

## What this run produced

Inside `run_dir`:

| Artifact | Judging criterion |
|---|---|
| `sft_data.jsonl`                          | Reproducible SFT corpus (heuristic teacher, rejection-sampled) |
| `sft/adapter/`                            | Post-SFT LoRA adapter (warm-start checkpoint) |
| `grpo/adapter/`                           | Final GRPO LoRA adapter |
| `reward_log.csv`                          | Per-episode total / grader / dense / format / steps |
| `training_dashboard.png`                  | **20 %** Showing improvement |
| `training_loss.png`                       | Min req — loss / KL / grad-norm |
| `eval_baseline.json` + `eval_trained.json`| **10 %** Reward & pipeline coherence |
| `eval_comparison.png`                     | **20 %** Multiple runs on the same axes |
| `trajectory_compare_<task>_seed<n>.png`   | **30 %** Storytelling (qualitative before/after) |

### Timing budget (T4)

| Stage | Estimate |
|---|---|
| Install + clone + smoke | ≈ 6 min |
| Load model (Unsloth 4-bit) | ≈ 3 min |
| SFT data gen (16 episodes) | ≈ 10 min |
| SFT training (1 epoch) | ≈ 4 min |
| Sanity check | ≈ 3 min |
| GRPO (24 ep × 4 gen × ≈ 22 turns) | ≈ 90–110 min |
| Plots + eval + trajectory | ≈ 12 min |
| **Total** | **≈ 130–150 min** ✓ |

### Iterating without touching the notebook

Every helper above lives in [`parkinsons_Motor/train.py`](../parkinsons_Motor/train.py): tweak `SYSTEM_PROMPT`, `DEFAULT_REWARD_WEIGHTS`, `heuristic_action`, or the per-task max-turn caps in cell 3, then re-run from cell 6. The same artifacts regenerate.
